In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import joblib


Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import joblib

In [3]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [4]:
PROJECT_ROOT = "/content/drive/MyDrive/heart_disease_project"
PROCESSED_DATA_PATH = PROJECT_ROOT + "/data/processed"
MODELS_PATH = PROJECT_ROOT + "/models"

In [5]:
X_train = np.load(PROCESSED_DATA_PATH + "/X_train.npy")
y_train = np.load(PROCESSED_DATA_PATH + "/y_train.npy")
X_test  = np.load(PROCESSED_DATA_PATH + "/X_test.npy")
y_test  = np.load(PROCESSED_DATA_PATH + "/y_test.npy")

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

Train shape: (814, 21)
Test shape : (184, 21)


In [6]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),
    "SVM": SVC(
        probability=True,
        random_state=42
    )
}

In [7]:
results = []

trained_models = {}

for name, model in models.items():
    # Train
    model.fit(X_train, y_train)
    trained_models[name] = model

    # Predict
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append([name, acc, f1, auc])

    print(f"{name} trained")

Logistic Regression trained
Decision Tree trained
Random Forest trained
SVM trained


In [8]:
results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "F1 Score", "ROC AUC"]
)

results_df.sort_values(by="ROC AUC", ascending=False)

,Model,Accuracy,F1 Score,ROC AUC
2,Random Forest,0.853261,0.868293,0.922286
3,SVM,0.847826,0.867925,0.917384
0,Logistic Regression,0.847826,0.864078,0.916906
1,Decision Tree,0.766304,0.788177,0.764108


In [9]:
best_model_name = results_df.sort_values(
    by="ROC AUC",
    ascending=False
).iloc[0]["Model"]

best_model = trained_models[best_model_name]

print("Best Model:", best_model_name)

Best Model: Random Forest


In [10]:
joblib.dump(
    best_model,
    MODELS_PATH + "/best_model.pkl"
)

print("Best model saved successfully.")

Best model saved successfully.


In [11]:
if best_model_name == "Random Forest":
    import matplotlib.pyplot as plt

    feature_importance = best_model.feature_importances_
    feature_names = joblib.load(PROJECT_ROOT + "/models/scaler.pkl").feature_names_in_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": feature_importance
    }).sort_values(by="Importance", ascending=False)

    importance_df

In [13]:
import os

In [14]:

assert os.path.exists(MODELS_PATH + "/best_model.pkl")
print("Modeling stage completed successfully.")

Modeling stage completed successfully.
